# 01 - Pirate LoRA toy (SmolLM2-360M-Instruct)

Learn LoRA by doing: take an open-weight instruct model, teach it pirate speak with a Low-Rank Adapter, and *see* the difference.

Pipeline in this notebook:

1. **Env check** (this section) - torch/HIP, precision decision
2. **Data** (issue #2) - `quill-voice/pirate`, 90/10 split, 5 holdout prompts
3. **Before + r=8 run** (issue #3) - baseline samples, first adapter
4. **r=32 run + compare + merge** (issue #4) - rank sweep, curves, `merge_and_unload`

How we work: read the explanation above each code cell *before* running it. If a cell surprises you, stop and ask - that surprise is the lesson.

## 0. Environment: why torch comes from AMD, not PyPI

PyPI `torch` wheels are CUDA/CPU-only - on our RX 7900 XTX (gfx1100) `torch.cuda.is_available()` would be `False` with them. So we install torch from AMD's wheel index (pinned in `scripts/install-rocm-torch.sh`). Run the cell below to prove the venv sees the GPU. If it prints the CPU fallback instead, re-run that script - everything after this cell assumes the printed precision decision (`bf16` on HIP, `fp32` on CPU).

In [ ]:
import platform
import torch
import matplotlib

print(f"python: {platform.python_version()}")
print(f"torch: {torch.__version__}")
hip = torch.cuda.is_available()
print(f"hip_available: {hip}")
if hip:
    print(f"device_count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"device[{i}]: {torch.cuda.get_device_name(i)}")
    use_bf16 = torch.cuda.is_bf16_supported()
else:
    print("device: CPU (fallback - see scripts/install-rocm-torch.sh)")
    use_bf16 = False
print(f"matplotlib: {matplotlib.__version__}")
print("precision decision:", "bf16" if use_bf16 else "fp32")

## 0b. Plotting smoke test

Later sections plot train/val loss, learning-rate schedule, and grad-norm with `matplotlib`. This cell proves plotting works in your setup - expect a sine wave, nothing more.

In [ ]:
import math
import matplotlib.pyplot as plt

xs = [i / 10 for i in range(100)]
ys = [math.sin(x) for x in xs]
plt.figure()
plt.plot(xs, ys)
plt.title("smoke test - real loss curves arrive in issues #3/#4")
plt.xlabel("step")
plt.ylabel("sin(step)")
plt.show()

## Next: data (issue #2)

Env proven. Next section (issue #2) loads `quill-voice/pirate` (797 rows), makes the 90/10 split (seed 42), freezes 5 holdout prompts, and tokenizes with `apply_chat_template` (max 512, assistant-only).